# Three Domains, One Pipeline

**Ontology-driven extraction and resolution for travel, customer service, and ecommerce — with no LLM
anywhere in the pipeline.**

The eight notebooks before this one build up a method on two corpora. This one asks whether the method
transfers: three new domains, three new ontologies, three corpora the pipeline has never seen, and the same
194M-parameter encoder doing all the work.

The constraint is the interesting part. **No LLM is used for extraction, resolution, classification or
adjudication.** Not because LLMs are bad at this — [notebook 03](03_extractors_head_to_head.ipynb) measured
Claude slightly ahead of GLiNER2.5 on triple recall — but because "no API key, no per-document cost, runs on
a laptop" is a genuine engineering position, and it is worth knowing exactly how far it goes.

It also forces the interesting techniques into view. Without a model that can be asked nicely, you need:

| domain | the hard problem | the technique that solves it |
|---|---|---|
| **Travel** | codes no string metric can match — `LHR` vs `London Heathrow` | a **gazetteer**: dictionary linking against a controlled vocabulary |
| **Customer service** | multi-turn threads where most subjects are pronouns | deterministic **coreference layers** + episode windowing |
| **Shopping** | near-duplicate product names that must *not* merge | **type-scoped blocking** and a precision-first resolver |

Plus three GLiNER2.5 capabilities the earlier notebooks never used: `extract_json` for structured records,
`RegexValidator` for pattern-constrained spans, and `classify_text` for document-level labels.

Everything lands in Neo4j and is drawn with NVL at the end.

## 0. Setup

In [1]:
import os, sys, time, json, warnings
from pathlib import Path

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT / "src"))
warnings.filterwarnings("ignore", category=FutureWarning)

import pandas as pd
pd.set_option("display.width", 200)
pd.set_option("display.max_colwidth", 46)

import kgx
from kgx.domains import TRAVEL, CUSTOMER_SERVICE, SHOPPING
from kgx.gazetteer import Gazetteer
from kgx.evaluate import score_triples, graph_triples, comparison_table, MatchPolicy

from kgx.data.travel import (DOCUMENTS as TRAVEL_DOCS, GOLD_TRAVEL_FACTS,
                             ALIAS_GROUPS as TRAVEL_ALIASES, IATA_GAZETTEER)
from kgx.data.customer_service import (THREADS, GOLD_CS_FACTS, GOLD_INTENTS,
                                       ALIAS_GROUPS as CS_ALIASES)
from kgx.data.shopping import (DOCUMENTS as SHOP_DOCS, GOLD_SHOPPING_FACTS, GOLD_ASPECTS,
                               ALIAS_GROUPS as SHOP_ALIASES, DISTINCT_PAIRS)

MATCH = MatchPolicy(normalize_names=True, use_aliases=True)

print(f"travel            {len(TRAVEL_DOCS):3} documents  {sum(len(d['text'].split()) for d in TRAVEL_DOCS):5} words"
      f"  {len(GOLD_TRAVEL_FACTS)} gold triples")
print(f"customer service  {len(THREADS):3} threads    {sum(len(t['turns']) for t in THREADS):5} turns"
      f"  {len(GOLD_CS_FACTS)} gold triples")
print(f"shopping          {len(SHOP_DOCS):3} documents  {sum(len(d['text'].split()) for d in SHOP_DOCS):5} words"
      f"  {len(GOLD_SHOPPING_FACTS)} gold triples")

travel             12 documents   2394 words  33 gold triples
customer service    8 threads       95 turns  40 gold triples
shopping           14 documents   2374 words  60 gold triples


In [2]:
extractor = kgx.GlinerExtractor()      # fastino/gliner2.5-base-v1, 194M, CPU
print(extractor)
print("\nno LLM client is imported anywhere in this notebook:")
print("  kgx.llm imported:", "kgx.llm" in sys.modules)

<GlinerExtractor fastino/gliner2.5-base-v1 194M params loaded in 4.9s>

no LLM client is imported anywhere in this notebook:
  kgx.llm imported: False


## 1. Three ontologies

An ontology here is the same object throughout the repo: node labels with annotation guidelines, edge types
with typed endpoints, and the legal `(head, RELATION, tail)` triples that fall out. It compiles to the
`JointSchema` GLiNER decodes against, and it also drives validation and the Cypher.

All three are held to 13 entity types and 16 relations. That ceiling is not tidiness — every type name and
description is serialised into the encoder input on *every call*, so a sprawling ontology costs context
budget and measurably costs recall.

In [3]:
DOMAINS = {"travel": TRAVEL, "customer service": CUSTOMER_SERVICE, "shopping": SHOPPING}

display(pd.DataFrame([
    {"domain": name,
     "entity types": len(o.entities),
     "relations": len(o.relations),
     "legal triple patterns": len(o.patterns()),
     "labels": ", ".join(o.entity_names[:6]) + " …"}
    for name, o in DOMAINS.items()
]).set_index("domain"))

for name, o in DOMAINS.items():
    print(f"\n── {name}")
    print("   " + ", ".join(o.relation_names))

,entity types,relations,legal triple patterns,labels
domain,,,,
travel,13,16,38,"traveller, airline, flight, airport, city,..."
customer service,13,16,33,"customer, agent, product, order, issue, re..."
shopping,13,16,47,"product, brand, product_category, retailer..."



── travel
   booked, covers, flies_on, operated_by, departs_from, arrives_at, connects_through, stays_at, located_in, member_of, serves, travels_for, holds_document, required_for, disrupted_by, replaces

── customer service
   reported_by, handled_by, escalated_to, concerns_product, about_order, owned_by, caused_by, component_of, resolved_by, replaces, compensated_with, refers_to_policy, breaches, contacted_via, has_tier, mentions_competitor

── shopping
   made_by, sold_by, belongs_to_category, has_aspect, priced_at, compatible_with, compares_to, complains_about, recommends, has_specification, bundled_with, replaces, covered_by, discounted_by, ships_with, offered_by


In [4]:
from IPython.display import Markdown, display as _d

core = TRAVEL.subset(
    entities=["traveller", "flight", "airline", "airport", "city", "hotel", "booking"],
    relations=["booked", "flies_on", "operated_by", "departs_from", "arrives_at",
               "stays_at", "located_in"],
    name="travel_core")
_d(Markdown("```mermaid\n" + core.to_mermaid() + "\n```"))

```mermaid
graph LR
  traveller([traveller])
  airline([airline])
  flight([flight])
  airport([airport])
  city([city])
  hotel([hotel])
  booking([booking])
  traveller -->|booked| booking
  traveller -->|flies_on| flight
  flight -->|operated_by| airline
  flight -->|departs_from| airport
  flight -->|arrives_at| airport
  traveller -->|stays_at| hotel
  airport -->|located_in| city
  hotel -->|located_in| city
  city -->|located_in| city
```

---

## 2. Travel — controlled vocabularies

Travel documents are full of identifiers: `LHR`, `MR117`, `MR7K2QX9`, `SKY-4471902`. They are the most
reliable information in the corpus and the least tractable for anything similarity-based. `LHR` and
`London Heathrow` share one letter.

Three techniques, in order of how much they buy.

### 2.1 Structured records — `extract_json`

A booking confirmation is a *form*. Asking for entities and relations and then reassembling the form is
backwards; GLiNER2.5 will fill the form directly. Fields are declared `name::type::description`, and the
description is doing the same annotation-guideline work as an entity label elsewhere.

In [5]:
BOOKING_RECORD = {"booking": [
    "reference::str::the booking confirmation code",
    "passenger::str::the traveller's name",
    "airline::str::the operating airline",
    "flight_number::str::the flight number",
    "origin::str::departure airport",
    "destination::str::arrival airport",
    "date::str::date of travel",
    "fare::str::the ticket price",
]}

confirmations = [d for d in TRAVEL_DOCS if "booking" in d["title"].lower()
                 or "confirm" in d["title"].lower()][:3]

# A single clean sentence first -- the case the feature is built for.
one_liner = ("Confirmation MR7K2QX9. Priya Raman flies Meridian Air MR117 from London Heathrow (LHR) "
             "to Singapore Changi (SIN) on 12 March 2026, fare GBP 812.40.")
display(pd.DataFrame(extractor.model.extract_json(one_liner, BOOKING_RECORD)["booking"]))

# Now the real documents, which are multi-line forms.
rows = []
for doc in confirmations:
    record = extractor.model.extract_json(doc["text"], BOOKING_RECORD)
    for filled in record.get("booking", []):
        rows.append({"doc_id": doc["doc_id"], **filled})
display(pd.DataFrame(rows))

,reference,passenger,airline,flight_number,origin,destination,date,fare
0,MR7K2QX9,Priya Raman,Meridian Air,MR117,London Heathrow (LHR),Singapore Changi (SIN),12 March 2026,GBP 812.40


,doc_id,reference,passenger,airline,flight_number,origin,destination,date,fare
0,t01,MR7K2QX9,Ms Priya Raman\nMeridian Skyline number: S...,Meridian Air,SKY-4471902 (Gold)\n\nOutbound — MR117,London Heathrow,"Singapore Changi (SIN), Terminal 1",12 March 2026,Platform Engineering travel budget
1,t01,MR7K2QX9,Ms Priya Raman,Meridian Air,SKY-4471902,London Heathrow,Singapore Changi,17 March 2026,Platform Engineering travel budget. Baggag...
2,t10,MR2H8VD6,John F. Kennedy,Meridian Air,MR310 Tue 02 Jun London Heathrow (LHR) T...,JFK,London Heathrow (LHR) T3 06:30 +1\n\nNonst...,06 Jun,"+1\n\nNonstop both ways, no connection, bo..."


On the single sentence, every field is right. One forward pass, no relation extraction, no resolution, no
assembly.

On the real documents it **bleeds**. `passenger` swallows the loyalty line that follows it, `flight_number`
picks up a loyalty number, `fare` returns a budget code. The failure is systematic and worth understanding:
these confirmations are multi-line forms where several plausible values sit near each field, separated by
newlines the model treats as ordinary whitespace, and nothing in the schema says where a field *ends*. A
field with one candidate in the text is extracted cleanly; a field with three candidates within a few tokens
is a coin toss.

So `extract_json` is not a general document parser. It is very good at pulling a record out of prose, and
poor at pulling one out of a layout — which is a statement about where the information lives, not about the
model. For a real booking pipeline the layout is regular enough to parse directly, and the model's job would
be the free-text parts that are not.

The rest of this section uses the graph path instead, and the comparison is the point: **when a document is a
form, parse the form; when it is prose, extract a graph.**

### 2.2 Pattern-constrained spans — `RegexValidator`

A booking reference has a shape. Telling the model the shape is cheaper than filtering afterwards, and
`RegexValidator` rejects any span that does not match before it reaches the output.

In [6]:
from gliner2 import RegexValidator

sample = confirmations[0]["text"]

loose = extractor.model.create_schema().entities(
    {"booking_reference": "an alphanumeric booking confirmation code"})
strict = extractor.model.create_schema().entities(
    {"booking_reference": "an alphanumeric booking confirmation code"},
    validators=[RegexValidator(r"^[A-Z]{2}[A-Z0-9]{4,8}$")])

print("without a validator:", extractor.model.extract(sample, loose).get("entities"))
print("with    a validator:", extractor.model.extract(sample, strict).get("entities"))

impossible = extractor.model.create_schema().entities(
    {"booking_reference": "an alphanumeric booking confirmation code"},
    validators=[RegexValidator(r"^ZZZ\d+$")])
print("with an impossible pattern:", extractor.model.extract(sample, impossible).get("entities"))

without a validator: {'booking_reference': ['MR7K2QX9', 'SKY-4471902', 'MR118', 'MR117']}
with    a validator: {'booking_reference': ['MR7K2QX9']}
with an impossible pattern: {'booking_reference': []}


The third line is the point: the validator genuinely filters rather than re-ranking. That makes it a hard
guarantee about the shape of what enters the graph — the same kind of guarantee typed endpoints give edges,
applied to span content.

### 2.3 The graph

In [7]:
t = time.time()
travel_graphs = extractor.extract_batch(TRAVEL_DOCS, TRAVEL)
travel_time = time.time() - t

print(f"{len(TRAVEL_DOCS)} documents in {travel_time:.1f}s")
print(f"{sum(len(g.mentions) for g in travel_graphs)} mentions, "
      f"{sum(len(g.edges) for g in travel_graphs)} edges")
print(f"ontology violations: {sum(len(g.validate(TRAVEL)) for g in travel_graphs)}")

from kgx.extract import mentions_frame
tm = mentions_frame(travel_graphs)
display(tm["type"].value_counts().to_frame("mentions").T)

12 documents in 7.1s
256 mentions, 186 edges
ontology violations: 0


type,city,flight,traveller,airport,country,airline,travel_document,disruption,booking,hotel,ground_transport,trip_purpose,loyalty_program
mentions,47,42,21,21,21,18,17,17,15,11,11,8,7


### 2.4 Gazetteer versus similarity

Now the resolution question. The corpus writes London Heathrow four ways and Meridian Air three ways. Two
approaches are available and they fail in opposite places.

`kgx.EntityResolver` compares mentions *to each other* — normalise, block, score, cluster. It knows nothing
in advance, which is what makes it work on the business-news corpus where nobody publishes a list of
companies.

`kgx.Gazetteer` compares mentions *to a published list*. Airports have IATA codes. That is not a fact about
this corpus, it is a fact about aviation, and using it turns an inference into a lookup.

In [8]:
gaz = Gazetteer(IATA_GAZETTEER)
print(gaz)
print("\nwhat a lookup returns, and by which rule:")
for surface in ["LHR", "Heathrow", "London Heathrow", "London Heathrow (LHR)",
                "Meridian Air", "MR", "Gatwick"]:
    print(f"  {surface!r:26} -> {gaz.lookup(surface)}")

<Gazetteer 12 entries, kinds=['airline', 'airport']>

what a lookup returns, and by which rule:
  'LHR'                      -> ('LHR', 'code', 1.0)
  'Heathrow'                 -> ('LHR', 'alias', 1.0)
  'London Heathrow'          -> ('LHR', 'name', 1.0)
  'London Heathrow (LHR)'    -> ('LHR', 'code', 1.0)
  'Meridian Air'             -> ('MR', 'name', 1.0)
  'MR'                       -> ('MR', 'code', 1.0)
  'Gatwick'                  -> None


In [9]:
travel_mentions = [m for g in travel_graphs for m in g.mentions]

# Only look up the types the vocabulary actually covers, and require the
# gazetteer kind to agree with the ontology type -- otherwise a city called
# "Meridian" would happily link to an airline.
LOOKUP_TYPES = ["airport", "airline", "city"]
KIND_OF = {"airport": "airport", "airline": "airline", "city": "city"}

report = gaz.coverage(travel_mentions, types=LOOKUP_TYPES, kinds=KIND_OF)
print(json.dumps(report, indent=1))

{
 "considered": 86,
 "linked": 32,
 "coverage": 0.372,
 "by_rule": {
  "name": 30,
  "code": 2
 },
 "distinct_keys": 5,
 "unlinked_surfaces": [
  "Changi",
  "Chiba",
  "Frankfurt",
  "Frankfurt am Main",
  "Heathrow",
  "Kestrel Cars",
  "London",
  "London Heathrow",
  "Meridian Air eleven",
  "New York",
  "Northwind Airways six",
  "Paddington"
 ]
}


In [10]:
# The kind constraint is a precision/recall dial. Measure it rather than assume it.
for label, kinds in [("type must match gazetteer kind", KIND_OF), ("no kind constraint", None)]:
    r = gaz.coverage(travel_mentions, types=LOOKUP_TYPES, kinds=kinds)
    print(f"  {label:32} linked {r['linked']:3}/{r['considered']}  "
          f"({r['coverage']:.0%})  distinct ids {r['distinct_keys']}")

# Which surfaces does the constraint reject, and were they right to reject?
strict_ok = {m.mention_id for m in gaz.resolve(travel_mentions, types=LOOKUP_TYPES, kinds=KIND_OF)[0]}
loose = gaz.resolve(travel_mentions, types=LOOKUP_TYPES)[0]
rejected = [m for m in loose if m.mention_id not in strict_ok]
by_id = {m.mention_id: m for m in travel_mentions}
print(f"\n{len(rejected)} links the kind constraint refused:")
seen = set()
for m in rejected:
    key = (by_id[m.mention_id].type, m.matched_surface, m.key)
    if key in seen: continue
    seen.add(key)
    print(f"    extracted as {by_id[m.mention_id].type:9} -> {m.key} ({gaz.entries[m.key].kind}): {m.matched_surface!r}")

  type must match gazetteer kind   linked  32/86  (37%)  distinct ids 5
  no kind constraint               linked  49/86  (57%)  distinct ids 7

17 links the kind constraint refused:
    extracted as city      -> SIN (airport): 'Changi'
    extracted as city      -> LHR (airport): 'Heathrow'
    extracted as city      -> FRA (airport): 'Frankfurt am Main'
    extracted as city      -> NRT (airport): 'Tokyo Narita'
    extracted as city      -> LHR (airport): 'London Heathrow'
    extracted as city      -> FRA (airport): 'Frankfurt'
    extracted as city      -> GRU (airport): 'São Paulo Guarulhos'


**On this corpus the constraint is pure loss.** It costs 20 points of coverage, and every one of the links it
refused was correct: seventeen refusals, all of them `city` mentions of a genuine airport — `Heathrow`,
`Frankfurt am Main`, `Tokyo Narita`, `São Paulo Guarulhos`. It caught nothing.

The cause is a taxonomy disagreement rather than a modelling error on either side. "Flying to London
Heathrow" *is* a place as much as a facility, so the extractor typing it `city` is defensible; the aviation
vocabulary calling it an airport is also defensible. They were written by different people for different
purposes and the constraint sits between them.

Which does not make the constraint wrong in general — it is what stops a `city` mention of "Meridian" linking
to Meridian Air, and that would be a worse error than a missed link. It makes it wrong *here*, and the honest
conclusion is that a kind constraint has to be validated against the extractor's actual type assignments
rather than adopted on principle. The fix is a one-line widening — let `city` also resolve to an airport —
and the notebook keeps the strict setting only so the measurement stays visible.

Coverage is reported over the mentions the vocabulary was actually asked about — airports, airlines and
cities — not over every span in the corpus. The unlinked list is the useful half of the output: those are
surfaces the corpus contains and the dictionary does not, and each one is either a gazetteer gap to fill or a
genuine miss to route to the similarity path. A gazetteer's quality *is* its coverage, and a miss should stay
a miss: silently snapping an unknown airport to the nearest known one destroys exactly the signal you wanted.

In [11]:
# Head to head on the same mentions: can each approach put every surface of one
# entity in one place?
from rapidfuzz import fuzz

resolver = kgx.EntityResolver(threshold=0.90)
resolution = resolver.resolve(travel_mentions)

def cluster_of(resolution, surface, type_):
    for m in travel_mentions:
        if m.text == surface and m.type == type_:
            return resolution.mention_to_canon.get(m.mention_id)
    return None

probes = [("airport", "LHR", "London Heathrow"),
          ("airline", "MR", "Meridian Air"),
          ("airport", "JFK", "John F. Kennedy")]

rows = []
for type_, code, name in probes:
    same_cluster = (cluster_of(resolution, code, type_) is not None
                    and cluster_of(resolution, code, type_) == cluster_of(resolution, name, type_))
    g_code, g_name = gaz.lookup(code), gaz.lookup(name)
    rows.append({
        "pair": f"{code} / {name}",
        "Jaro-Winkler": round(fuzz.WRatio(code, name) / 100, 3),
        "similarity resolver merges": same_cluster,
        "gazetteer links": (g_code is not None and g_name is not None
                            and g_code[0] == g_name[0]),
        "gazetteer key": g_code[0] if g_code else None,
    })
display(pd.DataFrame(rows).set_index("pair"))

,Jaro-Winkler,similarity resolver merges,gazetteer links,gazetteer key
pair,,,,
LHR / London Heathrow,0.45,False,True,LHR
MR / Meridian Air,0.60,False,True,MR
JFK / John F. Kennedy,0.45,False,True,JFK


That table is the argument for keeping both. Where a controlled vocabulary exists, use it: the link is exact,
instant, explainable, and it produces a **stable identifier** rather than a cluster — `airport:LHR` means the
same node next month, in a different corpus, and for an airport that appears in no document at all.

Where one does not exist — hotels, trip purposes, disruptions — the similarity resolver is the only option,
and the two compose cleanly: link what the dictionary covers, cluster the remainder.

In [12]:
matched, unmatched = gaz.resolve(travel_mentions, types=LOOKUP_TYPES, kinds=KIND_OF)
pinned = gaz.canonical_map(matched, prefix="gaz:")
# The dictionary knows the canonical name; the mentions only know surfaces.
pin_names = {f"gaz:{m.key}": m.canonical for m in matched}

# Everything the dictionary could not account for goes through the similarity path.
rest = resolver.resolve(unmatched)
travel_kg = kgx.build_graph(travel_graphs, rest, TRAVEL, pin=pinned, pin_names=pin_names)

print(f"{len(travel_mentions)} mentions")
print(f"  {len(matched):3} linked to the vocabulary  -> {len(set(pinned.values()))} stable ids")
print(f"  {len(unmatched):3} resolved by similarity  -> {len(rest.entities)} clusters")
print()
print(travel_kg.summary())

256 mentions
   32 linked to the vocabulary  -> 5 stable ids
  224 resolved by similarity  -> 105 clusters

KnowledgeGraph: 110 entities, 73 edges
  node labels: {'flight': 20, 'city': 14, 'booking': 12, 'disruption': 11, 'traveller': 10, 'travel_document': 8, 'ground_transport': 8, 'airport': 6, 'country': 5, 'hotel': 5, 'airline': 5, 'trip_purpose': 3, 'loyalty_program': 3}
  edge types:  {'located_in': 17, 'operated_by': 8, 'stays_at': 8, 'required_for': 5, 'travels_for': 5, 'departs_from': 4, 'arrives_at': 4, 'connects_through': 4, 'serves': 3, 'flies_on': 3, 'replaces': 3, 'booked': 2, 'member_of': 2, 'disrupted_by': 2, 'holds_document': 2, 'covers': 1}


In [13]:
travel_score = score_triples(graph_triples(travel_kg), GOLD_TRAVEL_FACTS,
                             policy=MATCH, aliases=TRAVEL_ALIASES)
print(travel_score.report(limit=5))

precision 0.233  recall 0.515  f1 0.321   (17/33 gold, 73 predicted, policy: normalised + aliases)

  missed (16):
    Tomas Ferreira -booked-> NW3F8LT2
    Dr. Elena Vasquez -booked-> NWQ9ZP41
    MR7K2QX9 -covers-> MR117
    NWQ9ZP41 -covers-> Narita Express
    Dr. Elena Vasquez -flies_on-> NW612

  spurious (56):
    MR118 -departs_from-> London Heathrow
    Changi -located_in-> Singapore
    Meridian Air -serves-> Changi
    MR204 -arrives_at-> Frankfurt am Main
    NW882 São Paulo Guarulhos -connects_through-> Frankfurt am Main


Read recall, not precision. `GOLD_TRAVEL_FACTS` is a *representative sample* covering all sixteen relations,
not an exhaustive enumeration — the corpus supports roughly half again as many true triples. So a large share
of the "spurious" list is true-and-unlabelled, and precision here is a lower bound rather than an estimate,
exactly as in notebook 03. The same caveat applies to all three domains in this notebook.

---

## 3. Customer service — conversational structure

Support threads break the assumptions a document extractor makes. The subject of most facts is a pronoun, the
speaker alternates, and the thread as a whole carries labels — intent, priority, whether it was resolved —
that no individual span expresses.

[Notebook 08](08_coreference.ipynb) measured four neural coreference engines against four deterministic rules
on a different conversational corpus and the rules won. This is the replication on unseen data.

In [14]:
print(f"{len(THREADS)} threads, {sum(len(t['turns']) for t in THREADS)} turns\n")
display(pd.DataFrame([
    {"thread": t["thread_id"], "channel": t["channel"], "turns": len(t["turns"]),
     "subject": t["subject"][:58]} for t in THREADS
]).set_index("thread"))

print("\nfirst three turns of t01:")
for turn in THREADS[0]["turns"][:3]:
    print(f"  {turn['speaker']:8}: {turn['text'][:120]}")

8 threads, 95 turns



,channel,turns,subject
thread,,,
t01,email,12,Aurora 14 shutting down at 40% battery
t02,chat,13,left Halcyon Bud keeps dropping out
t03,phone log,12,"N600 ordered on the tenth, still not shipped"
t04,email,11,Charged twice for Priority Care on 28 March
t05,chat,14,"replacement N600 is dead, want the money back"
t06,email,10,Fourth request: refund RF-2261 has still n...
t07,chat,11,cancel Priority Care and order F-88132
t08,chat,12,charging case only charges the right earbud



first three turns of t01:
  customer: I bought an Aurora 14 in April last year — order A-99213 — and for about three weeks now it has been shutting off withou
  agent   : Hi Priya, this is Dev Shah in hardware support. Thank you for the detail — that is more than most people give me. A hard
  customer: Design capacity 71.2 Wh. Full charge capacity 39.8 Wh. Cycle count 214.


### 3.1 What raw threads cost

In [15]:
from collections import Counter

def relations_found(graphs, ontology):
    seen = Counter(e.type for g in graphs for e in g.edges)
    return seen, [r for r in ontology.relation_names if r not in seen]

# (a) raw: speaker labels prepended, nothing else
raw_docs = [{"doc_id": t["thread_id"],
             "text": "\n".join(f"{x['speaker']}: {x['text']}" for x in t["turns"])}
            for t in THREADS]
raw_graphs = extractor.extract_batch(raw_docs, CUSTOMER_SERVICE)
raw_seen, raw_missing = relations_found(raw_graphs, CUSTOMER_SERVICE)
raw_junk = sum(1 for g in raw_graphs for m in g.mentions
               if m.text.strip().lower() in {"customer", "agent"})

print(f"raw threads          {sum(len(g.mentions) for g in raw_graphs):4} mentions "
      f"{sum(len(g.edges) for g in raw_graphs):4} edges")
print(f"  relations that never fired : {raw_missing}")
print(f"  spans that are just a speaker label: {raw_junk}")

/var/folders/rd/xd8h_pz55g3681m7klj4c17r0000gp/T/ipykernel_22171/2181954179.py:11: RuntimeWarning: input is 520 words with 16 relation types and no windowing; relation recall degrades sharply past ~400 words and can lose most of a document's relations. Pass config=JointIEConfig(max_len=512), use extract_long(), or split into episodes.
  raw_graphs = extractor.extract_batch(raw_docs, CUSTOMER_SERVICE)


raw threads           195 mentions  196 edges
  relations that never fired : ['handled_by', 'caused_by', 'replaces']
  spans that are just a speaker label: 60


Two failures, both structural. Four relations never fire at all, and sixty spans are the literal words
"customer" and "agent" — the labels the rendering inserted, extracted as entities. A graph built from this
has a `customer` node that is not a customer.

### 3.2 Preprocessing, and the length cliff

`ConversationPreprocessor` rewrites first-person references to the speaker's name, expands unambiguous first
names, and binds sentence-initial pronouns to a recent antecedent. Then the thread is split into overlapping
**episodes** rather than extracted whole — notebook 01 §11 measured relation recall collapsing on long
conversational input, and a support thread is exactly that shape.

In [16]:
CUSTOMERS = {"t01": "Priya Raman", "t02": "Marcus Webb", "t03": "Ana Duarte",
             "t04": "Priya Raman", "t05": "Marcus Webb", "t06": "Ana Duarte",
             "t07": "Priya Raman", "t08": "Marcus Webb"}
ROSTER = ["Priya Raman", "Marcus Webb", "Ana Duarte", "Dev Shah", "Tomas Ferreira"]

def prepare(threads, *, window=None):
    """Render threads as extraction units, optionally split into episodes."""
    docs, preprocessors = [], {}
    for t in threads:
        pre = kgx.ConversationPreprocessor(
            CUSTOMERS.get(t["thread_id"], "the customer"),
            assistant_name="Support Agent", roster=ROSTER)
        preprocessors[t["thread_id"]] = pre
        session = {"session_id": t["thread_id"], "timestamp": t["opened"],
                   "turns": [{"speaker": "user" if x["speaker"] == "customer" else "assistant",
                              "text": x["text"]} for x in t["turns"]]}
        if window is None:
            r = pre.render(session)
            docs.append({"doc_id": t["thread_id"], "text": r.text, "thread_id": t["thread_id"]})
        else:
            docs.extend(pre.episodes([session], window=window, stride=window - 1))
    return docs, preprocessors

rows = []
variants = {}
for label, window in [("preprocessed, whole thread", None), ("preprocessed, 4-turn episodes", 4)]:
    docs, pres = prepare(THREADS, window=window)
    with warnings.catch_warnings():
        # The whole-thread variant trips the long-input guard on purpose; the
        # table below is the measurement, so the warning is noise here.
        warnings.simplefilter("ignore", RuntimeWarning)
        graphs = extractor.extract_batch(docs, CUSTOMER_SERVICE)
    dropped = sum(p.clean_mentions([g for g in graphs]) for p in list(pres.values())[:1])
    seen, missing = relations_found(graphs, CUSTOMER_SERVICE)
    variants[label] = (docs, graphs)
    rows.append({"input": label, "units": len(docs),
                 "mentions": sum(len(g.mentions) for g in graphs),
                 "edges": sum(len(g.edges) for g in graphs),
                 "relations never fired": len(missing)})

rows.insert(0, {"input": "raw threads", "units": len(raw_docs),
                "mentions": sum(len(g.mentions) for g in raw_graphs),
                "edges": sum(len(g.edges) for g in raw_graphs),
                "relations never fired": len(raw_missing)})
display(pd.DataFrame(rows).set_index("input"))

,units,mentions,edges,relations never fired
input,,,,
raw threads,8,195,196,3
"preprocessed, whole thread",8,216,206,4
"preprocessed, 4-turn episodes",32,431,387,2


The same shape notebook 01 found on a different corpus: preprocessing the whole thread *removes* the junk
spans but does not help relation recall, because the thread is still one long input. Windowing is what
recovers the relations.

That is a replication rather than a restatement — the ordering was established on the agent-memory corpus and
holds on support threads written for a different ontology.

In [17]:
cs_docs, cs_graphs = variants["preprocessed, 4-turn episodes"]
seen, cs_missing = relations_found(cs_graphs, CUSTOMER_SERVICE)
print(f"relations still never fired: {cs_missing}")
print(f"ontology violations: {sum(len(g.validate(CUSTOMER_SERVICE)) for g in cs_graphs)}")
print(f"\ntop relations: {dict(seen.most_common(8))}")

relations still never fired: ['handled_by', 'replaces']
ontology violations: 0

top relations: {'owned_by': 132, 'contacted_via': 92, 'compensated_with': 65, 'mentions_competitor': 42, 'component_of': 15, 'concerns_product': 14, 'reported_by': 8, 'escalated_to': 5}


Three relations never fire at any setting. That is the ontology declaring something the corpus does not say
in a form the extractor can find — the same failure notebook 02 hit when `SUPPLIES` and `SUBJECT_TO` were
declared and never produced. An ontology is a hypothesis about the text, and part of what extraction does is
test it.

### 3.3 Document-level labels — `classify_text`

Intent and priority are properties of a thread, not of a span. GLiNER2.5 classifies against a label set in
the same forward-pass framework as everything else, and the corpus ships gold labels for all eight threads.

In [18]:
INTENTS = ["refund request", "technical support", "order status",
           "billing dispute", "complaint", "cancellation"]
PRIORITIES = ["low", "medium", "high", "urgent"]

rows = []
for t in THREADS:
    text = "\n".join(f"{x['speaker']}: {x['text']}" for x in t["turns"])
    out = extractor.model.classify_text(
        text, {"intent": INTENTS, "priority": PRIORITIES}, include_confidence=True)
    gold = GOLD_INTENTS[t["thread_id"]]
    rows.append({
        "thread": t["thread_id"],
        "intent": out["intent"]["label"], "gold intent": gold["intent"],
        "intent ok": out["intent"]["label"] == gold["intent"],
        "conf": round(out["intent"]["confidence"], 2),
        "priority": out["priority"]["label"], "gold priority": gold["priority"],
        "priority ok": out["priority"]["label"] == gold["priority"],
    })
clf = pd.DataFrame(rows).set_index("thread")
display(clf)
print(f"intent accuracy   {clf['intent ok'].mean():.0%}  ({clf['intent ok'].sum()}/{len(clf)})")
print(f"priority accuracy {clf['priority ok'].mean():.0%}  ({clf['priority ok'].sum()}/{len(clf)})")

,intent,gold intent,intent ok,conf,priority,gold priority,priority ok
thread,,,,,,,
t01,refund request,technical support,False,0.92,high,high,True
t02,technical support,technical support,True,0.87,high,medium,False
t03,cancellation,order status,False,0.96,high,low,False
t04,refund request,billing dispute,False,0.75,high,medium,False
t05,refund request,refund request,True,0.99,high,high,True
t06,refund request,complaint,False,0.87,high,urgent,False
t07,cancellation,cancellation,True,1.00,low,medium,False
t08,complaint,technical support,False,0.80,high,low,False


intent accuracy   38%  (3/8)
priority accuracy 25%  (2/8)


**This does not work, and the way it fails is the useful part.**

Intent lands 3 of 8 against a 6-class chance baseline of 1 in 6 — better than guessing, not by much, and on
eight threads a single flip is 12.5 points, so the gap from chance is not distinguishable from noise.
Priority lands 2 of 8 against a 4-class baseline of 1 in 4, which *is* chance.

Look at the priority column rather than its score. The model answers `high` for seven of the eight threads.
It is very close to a constant predictor, and a constant predictor scores 2 of 8 here because three threads
happen to be high. Notebook 03 hit the same trap on an inference probe: a system that always emits the same
answer looks competent whenever the majority class is the right one, and only a distribution reveals it.

The reason is worth stating because it generalises. Intent is *written down* — "I want a refund, not a
replacement" is a span. Priority is a judgement about business impact that nobody types into a support
thread, and there is nothing in the text for a span-based classifier to key on. Confidences stay high
throughout (0.75–1.00), so calibration will not rescue it either.

This is the boundary of the no-LLM position, and it is a real one. Document-level labels that require
weighing severity are the first thing worth escalating; unlike the extraction tasks in §2 and §4, no amount
of ontology design fixes it.

### 3.4 The graph

In [19]:
cs_mentions = [m for g in cs_graphs for m in g.mentions]
cs_resolution = kgx.EntityResolver(threshold=0.90).learn_aliases(
    d["text"] for d in cs_docs).resolve(cs_mentions)
cs_kg = kgx.build_graph(cs_graphs, cs_resolution, CUSTOMER_SERVICE)
print(cs_kg.summary())

cs_score = score_triples(graph_triples(cs_kg), GOLD_CS_FACTS, policy=MATCH, aliases=CS_ALIASES)
print(f"\n{cs_score.report(limit=4)}")

long_heads = [t for t in GOLD_CS_FACTS if len(t[0].split()) > 4]
print(f"\ngold triples whose head is a descriptive phrase rather than a name: "
      f"{len(long_heads)}/{len(GOLD_CS_FACTS)}")
for t in long_heads[:3]:
    print(f"    {t[0]!r}")

KnowledgeGraph: 151 entities, 84 edges
  node labels: {'issue': 22, 'component': 20, 'product': 18, 'refund': 16, 'order': 13, 'sla': 11, 'resolution': 11, 'policy': 10, 'channel': 9, 'account_tier': 8, 'agent': 6, 'competitor': 4, 'customer': 3}
  edge types:  {'owned_by': 17, 'component_of': 12, 'concerns_product': 12, 'contacted_via': 11, 'compensated_with': 8, 'reported_by': 7, 'mentions_competitor': 4, 'breaches': 3, 'escalated_to': 3, 'about_order': 2, 'caused_by': 2, 'resolved_by': 1, 'has_tier': 1, 'refers_to_policy': 1}

precision 0.107  recall 0.225  f1 0.145   (9/40 gold, 84 predicted, policy: normalised + aliases)

  missed (31):
    Aurora 14 shuts down at 40% charge -reported_by-> Priya Raman
    left Halcyon Bud drops out -reported_by-> Marcus Webb
    charging case will not charge left earbud -reported_by-> Ana Duarte
    Aurora 14 shuts down at 40% charge -handled_by-> Dev Shah

  spurious (75):
    Priya Raman -about_order-> A-99213
    shutdown fault -breaches-> 12-m

The score is low and part of it is a gold-design problem rather than an extraction problem. A third of the
gold triples have a *descriptive phrase* as their head — `"Aurora 14 shuts down at 40% charge"` — which is a
summary of an issue, not a span anyone wrote. The extractor produces `shutdown fault`, which is arguably a
better node name and scores as a miss and a spurious edge simultaneously.

This is the same caveat notebook 03 raised from the other direction: gold sets are hand-written, and where a
gold endpoint is not a surface in the text, no extractor can match it. The recall number here is a lower
bound on a corpus whose gold labels were written for readability.

In [20]:
# Two threads offer a replacement and then agree a refund instead, and `replaces`
# is in the ontology so the supersession could be extracted rather than inferred.
supersessions = [e for e in cs_kg.edges if e.type == "replaces"]
print(f"`replaces` edges found: {len(supersessions)}")
for e in sorted(supersessions, key=lambda x: -x.confidence)[:4]:
    print(f"  {cs_kg.name(e.head)!r} replaces {cs_kg.name(e.tail)!r}  conf={e.confidence:.2f}")

# What the corpus actually says, for comparison.
print("\nthe turns the relation was supposed to catch:")
for t in THREADS:
    for turn in t["turns"]:
        low = turn["text"].lower()
        if "instead of" in low or "rather than a replacement" in low or "refund instead" in low:
            print(f"  [{t['thread_id']}] {turn['text'][:130]}")

`replaces` edges found: 0

the turns the relation was supposed to catch:
  [t04] Three to five working days under reference RF-3390. If it is not there on the fifth working day, reply to this email and I will ra
  [t06] Both of those are on us. I have re-issued the refund against the replacement card as RF-2261-B, and I am escalating this to Tomas 
  [t08] It clicks. The LED does one long blink instead of two.


Zero. The corpus was written with two threads where a replacement is offered and a refund agreed instead, the
ontology declares `replaces(resolution -> resolution)`, and the extractor finds none of it.

That makes three relations in this ontology that never fire — `handled_by`, `replaces`, and the third from
§3.2. Compare notebook 01, where the *same* relation name on the agent-memory ontology fired at 0.99
confidence and drove the temporal supersession. The difference is what the text does with it: there, a user
says "I've switched to pnpm" — one clause, two named tools, an explicit substitution verb. Here the
supersession is spread across a negotiation ("we can send a replacement" … four turns later … "let's just do
the refund"), with the two resolutions never adjacent and the substitution never stated.

`replaces` is not a hard relation. It is a hard relation *in this shape of text*, and an ontology that works
on one corpus can quietly declare a relation the next corpus expresses in a way the extractor cannot see.

---

## 4. Shopping — near-duplicates and aspect opinion

Ecommerce inverts the resolution problem. In travel the difficulty was linking surfaces that look *nothing*
alike; here it is keeping apart surfaces that look almost identical and are different products.

`Aurora 14` and `Aurora 14 Pro` differ by one token. `Northwind Router N600` and `N600X` differ by one
character. Merging them corrupts every review, price and specification attached to either. Meanwhile the same
product genuinely appears as `Aurora 14`, `Aurora-14`, `Aurora14` and `the Aurora 14 laptop`, all of which
must collapse.

The corpus ships both lists, so this is scored rather than asserted.

In [21]:
print(f"{len(SHOP_DOCS)} documents: " +
      ", ".join(f"{k}×{sum(1 for d in SHOP_DOCS if d['kind']==k)}"
                for k in sorted({d["kind"] for d in SHOP_DOCS})))
print(f"\n{len(DISTINCT_PAIRS)} pairs that must NOT merge:")
for a, b in DISTINCT_PAIRS[:6]:
    print(f"    {a!r:26} vs {b!r}")
print(f"\n{len(SHOP_ALIASES)} alias groups that MUST merge, e.g.:")
for canon, variants in list(SHOP_ALIASES.items())[:3]:
    print(f"    {canon!r:26} <- {variants}")

14 documents: listing×4, qa×3, review×6, spec_sheet×1

12 pairs that must NOT merge:
    'Aurora 14'                vs 'Aurora 14 Pro'
    'Northwind Router N600'    vs 'Northwind Router N600X'
    'Halcyon Buds'             vs 'Halcyon Buds Pro'
    'Aurora 14'                vs 'Aurora Dock'
    'Aurora'                   vs 'Aurora 14'
    'Halcyon'                  vs 'Halcyon Buds'

15 alias groups that MUST merge, e.g.:
    'Aurora 14'                <- ['Aurora 14', 'Aurora-14', 'Aurora14', 'the Aurora 14 laptop', 'the base model']
    'Aurora 14 Pro'            <- ['Aurora 14 Pro', 'the Aurora 14 Pro', 'the Pro']
    'Aurora Dock'              <- ['Aurora Dock', 'the Aurora Dock', 'the Dock']


In [22]:
t = time.time()
shop_graphs = extractor.extract_batch(SHOP_DOCS, SHOPPING)
shop_time = time.time() - t
shop_mentions = [m for g in shop_graphs for m in g.mentions]

print(f"{len(SHOP_DOCS)} documents in {shop_time:.1f}s")
print(f"{len(shop_mentions)} mentions, {sum(len(g.edges) for g in shop_graphs)} edges")
print(f"ontology violations: {sum(len(g.validate(SHOPPING)) for g in shop_graphs)}")

14 documents in 22.8s
318 mentions, 538 edges
ontology violations: 0


### 4.1 The precision trap, measured

In [23]:
from rapidfuzz import fuzz

# How similar are the pairs that must stay apart, compared with the ones that must merge?
must_split = [(a, b, fuzz.WRatio(a, b) / 100) for a, b in DISTINCT_PAIRS]
must_merge = [(c, v, fuzz.WRatio(c, v) / 100)
              for c, vs in SHOP_ALIASES.items() for v in vs if v != c]

print(f"pairs that must SPLIT: similarity {min(s for *_, s in must_split):.2f}"
      f" – {max(s for *_, s in must_split):.2f}  (n={len(must_split)})")
print(f"pairs that must MERGE: similarity {min(s for *_, s in must_merge):.2f}"
      f" – {max(s for *_, s in must_merge):.2f}  (n={len(must_merge)})")
overlap = max(s for *_, s in must_split) >= min(s for *_, s in must_merge)
print(f"\nranges overlap: {overlap}  -> "
      f"{'no single string threshold separates them' if overlap else 'a threshold could separate them'}")
display(pd.DataFrame(sorted(must_split, key=lambda r: -r[2])[:6],
                     columns=["a", "b", "string similarity"]))

pairs that must SPLIT: similarity 0.27 – 0.98  (n=12)
pairs that must MERGE: similarity 0.21 – 0.95  (n=27)

ranges overlap: True  -> no single string threshold separates them


,a,b,string similarity
0,Northwind Router N600,Northwind Router N600X,0.976744
1,12-month manufacturer warranty,24-month manufacturer warranty,0.966667
2,Aurora 14,Aurora 14 Pro,0.950000
3,Halcyon Buds,Halcyon Buds Pro,0.950000
4,Aurora,Aurora 14,0.900000
5,Halcyon,Halcyon Buds,0.900000


The ranges overlap, so string similarity alone cannot do this job at any threshold — the same structural
finding notebook 05 reached from the opposite direction, where no threshold could *reach* the acronym pairs.

The hope is that context breaks the tie: a review of the Pro talks about a different price, a different
screen and a different weight, and `kgx.EntityResolver` embeds `"<type>: <name> || <local context>"` rather
than the bare name precisely so that signal is available. The next cell tests whether it is enough.

In [24]:
resolver = kgx.EntityResolver(threshold=0.90).learn_aliases(d["text"] for d in SHOP_DOCS)
shop_resolution = resolver.resolve(shop_mentions)

def cluster_for(surface):
    ids = {shop_resolution.mention_to_canon[m.mention_id]
           for m in shop_mentions
           if m.text.strip().casefold() == surface.casefold()
           and m.mention_id in shop_resolution.mention_to_canon}
    return ids

# Precision: did any pair that must stay apart get merged?
violations = []
for a, b in DISTINCT_PAIRS:
    shared = cluster_for(a) & cluster_for(b)
    if shared:
        violations.append((a, b, sorted(shared)[0]))

# Recall: did each alias group land in one cluster?
merged_ok, split_up = [], []
for canon, variants in SHOP_ALIASES.items():
    ids = set()
    for v in variants:
        ids |= cluster_for(v)
    (merged_ok if len(ids) <= 1 else split_up).append((canon, len(ids)))

print(f"must-split pairs wrongly merged : {len(violations)}/{len(DISTINCT_PAIRS)}")
for a, b, cid in violations:
    print(f"    {a!r} + {b!r} -> {cid}")
print(f"\nalias groups fully merged       : {len(merged_ok)}/{len(SHOP_ALIASES)}")
for canon, n in split_up[:6]:
    print(f"    {canon!r} split across {n} clusters")

must-split pairs wrongly merged : 3/12
    'Aurora 14' + 'Aurora 14 Pro' -> product:aurora-14-pro
    'Northwind Router N600' + 'Northwind Router N600X' -> product:northwind-router-n600x
    'Halcyon Buds' + 'Halcyon Buds Pro' -> product:halcyon-buds-pro

alias groups fully merged       : 6/15
    'Aurora 14' split across 2 clusters
    'Aurora Dock' split across 2 clusters
    'Halcyon Buds' split across 3 clusters
    'Northwind Router N600' split across 2 clusters
    'Aurora' split across 2 clusters
    'Northwind' split across 2 clusters


### It is not enough. The resolver fails in both directions.

Three of the twelve must-split pairs merged — `Aurora 14` with `Aurora 14 Pro`, `N600` with `N600X`,
`Halcyon Buds` with `Halcyon Buds Pro`. Every one is a real product collapsed into its own successor, taking
that successor's price, specifications and reviews with it.

And only six of fifteen alias groups fully merged, so the same resolver is *also* splitting products that
should be one node.

This is worth sitting with, because it contradicts the impression the earlier notebooks leave. Notebook 01
and notebook 05 both report B-cubed **precision 1.000** — nothing wrong was ever merged — on the business-news
corpus. That was not a property of the resolver. It was a property of a corpus in which no two distinct
entities had near-identical names. Give it `Aurora 14` and `Aurora 14 Pro` and the same code merges them at
the same threshold.

The mechanism is visible in the numbers above: the merged pairs score 0.95–0.98 on string similarity, the
context embedding contributes a fraction of the weighted score, and one token of difference cannot outvote
everything the two names share. Worse, the token that distinguishes them — `Pro`, `X` — is exactly the kind
of short suffix that normalisation is designed to be insensitive to.

Two fixes, and neither is a threshold change. A **model-number gazetteer**, as in §2, would make this exact:
product SKUs are a controlled vocabulary and a catalogue is a published list. Failing that, a rule that treats
a trailing model qualifier as a hard discriminator — if two candidate names differ only by a suffix in
`{Pro, Max, X, Plus, Mini}`, refuse the merge regardless of score. The first is better and the repo's own
§2 already demonstrates it; this section deliberately runs without it so the failure is on the record.

### 4.2 Aspect-level opinion — span attributes

In [25]:
from gliner2 import AttributeGroup

# A review is only useful if you know which aspect the sentiment attaches to.
# JointSchema edges carry no properties, so polarity is recovered as a span
# attribute and stored on the node -- the same two-pass shape notebook 01 used
# for modality on financial risk.
reviews = [d for d in SHOP_DOCS if d["kind"] == "review"]

shop_graphs_q = [
    extractor.qualify(g, {"sentiment": ["positive", "negative", "mixed", "neutral"]},
                      applies_to={"sentiment": ["product_aspect", "complaint"]})
    for g in shop_graphs
]

rows = [{"doc_id": g.doc_id, "aspect": m.text,
         "sentiment": m.attrs["sentiment"]["label"],
         "conf": round(m.attrs["sentiment"]["confidence"], 2)}
        for g in shop_graphs_q for m in g.mentions if "sentiment" in m.attrs]
aspects = pd.DataFrame(rows)
print(f"{len(aspects)} aspect mentions carry a polarity")
display(aspects.head(12))

26 aspect mentions carry a polarity


,doc_id,aspect,sentiment,conf
0,s02,screen,positive,0.82
1,s02,Build quality,positive,0.74
2,s02,Battery life,negative,0.34
3,s02,Shipping,positive,0.80
4,s02,Customer support,negative,0.39
5,s03,Battery life,positive,0.63
6,s03,keyboard,negative,0.89
7,s03,typos,negative,0.77
8,s03,Build quality,neutral,0.39
9,s03,keyboard,negative,0.90


In [26]:
# Score against the gold aspect list, matching on (doc_id, aspect) after
# normalisation -- gold names an aspect "screen", the model may say "OLED display".
from kgx.resolve import normalize

gold_lookup = {(g["doc_id"], normalize(g["aspect"]).key): g["sentiment"] for g in GOLD_ASPECTS}
pred_lookup = {(r["doc_id"], normalize(r["aspect"]).key): r["sentiment"]
               for r in rows}

shared = set(gold_lookup) & set(pred_lookup)
agree = sum(1 for k in shared if gold_lookup[k] == pred_lookup[k])
print(f"gold aspect mentions            : {len(gold_lookup)}")
print(f"model aspect mentions           : {len(pred_lookup)}")
print(f"aspects matched by exact name   : {len(shared)}")
print(f"  of those, polarity agrees     : {agree}/{len(shared)}"
      + (f"  ({agree/len(shared):.0%})" if shared else ""))
print("\nWhere they disagree:")
for k in sorted(shared):
    if gold_lookup[k] != pred_lookup[k]:
        print(f"    [{k[0]}] {k[1]!r}: gold={gold_lookup[k]} model={pred_lookup[k]}")

gold aspect mentions            : 29
model aspect mentions           : 22
aspects matched by exact name   : 11
  of those, polarity agrees     : 11/11  (100%)

Where they disagree:


Where gold and model name the same aspect they agree on polarity every time — 11 of 11. But the overlap is
the number to look at, not the agreement rate. Gold and model largely name *different* aspects
— gold says `screen`, the model extracts `OLED display`; gold says `build quality`, the model extracts
`chassis`. Both are defensible aspect names and an exact-name join sees neither.

That is a measurement problem this notebook does not solve, and it is worth being explicit that the agreement
rate is computed over whatever happens to align rather than over the aspects that matter. Aspect-based
sentiment needs either a fixed aspect taxonomy the extractor is constrained to, or a matcher that knows
`OLED display` is a kind of `screen` — which is an ontology, and would be the right next iteration.

Where the polarities *do* line up, they line up well, which is consistent with the direct probe: on a
sentence like *"the battery life is superb but the trackpad feels cheap"* the attribute pass attaches
`positive` and `negative` to the right spans.

In [27]:
shop_kg = kgx.build_graph(shop_graphs_q, shop_resolution, SHOPPING)
print(shop_kg.summary())

shop_score = score_triples(graph_triples(shop_kg), GOLD_SHOPPING_FACTS,
                           policy=MATCH, aliases=SHOP_ALIASES)
print(f"\n{shop_score.report(limit=4)}")

KnowledgeGraph: 135 entities, 133 edges
  node labels: {'specification': 21, 'product_aspect': 21, 'product': 20, 'price': 13, 'accessory': 10, 'complaint': 9, 'product_category': 8, 'promotion': 7, 'review_author': 7, 'brand': 6, 'shipping_option': 5, 'retailer': 4, 'warranty': 4}
  edge types:  {'has_aspect': 30, 'has_specification': 19, 'made_by': 12, 'sold_by': 12, 'priced_at': 8, 'belongs_to_category': 7, 'offered_by': 6, 'compares_to': 6, 'covered_by': 6, 'bundled_with': 6, 'recommends': 5, 'replaces': 5, 'discounted_by': 4, 'compatible_with': 4, 'complains_about': 2, 'ships_with': 1}

precision 0.195  recall 0.433  f1 0.269   (26/60 gold, 133 predicted, policy: normalised + aliases)

  missed (34):
    Aurora 14 -made_by-> Aurora
    Aurora 14 -sold_by-> Brightline Direct
    Halcyon Buds -sold_by-> Tidewater Supply
    Aurora 14 -belongs_to_category-> ultrabooks

  spurious (107):
    Aurora 14 Pro -belongs_to_category-> laptops
    Aurora 14 Pro -belongs_to_category-> ultraboo

---

## 5. One pipeline, three domains

Same model, same schema compiler, same resolver, same graph builder. The only things that changed are the
ontology and the pre-processing.

In [28]:
GRAPHS = {"travel": (TRAVEL, travel_kg, travel_score, travel_graphs),
          "customer service": (CUSTOMER_SERVICE, cs_kg, cs_score, cs_graphs),
          "shopping": (SHOPPING, shop_kg, shop_score, shop_graphs_q)}

rows = []
for name, (onto, kg, score, graphs) in GRAPHS.items():
    fired = {e.type for g in graphs for e in g.edges}
    rows.append({
        "domain": name,
        "docs": len(graphs),
        "mentions": sum(len(g.mentions) for g in graphs),
        "entities": len(kg.entities),
        "edges": len(kg.edges),
        "relations fired": f"{len(fired)}/{len(onto.relations)}",
        "ontology violations": sum(len(g.validate(onto)) for g in graphs),
        "gold recall": round(score.recall, 3),
        "gold F1": round(score.f1, 3),
    })
display(pd.DataFrame(rows).set_index("domain"))

,docs,mentions,entities,edges,relations fired,ontology violations,gold recall,gold F1
domain,,,,,,,,
travel,12,256,110,73,16/16,0,0.515,0.321
customer service,32,431,151,84,14/16,0,0.225,0.145
shopping,14,318,135,133,16/16,0,0.433,0.269


Three things hold across all three domains, and they are the transferable part.

**Zero ontology violations, everywhere.** 284 edges, three ontologies written by different hands, and not one
edge whose endpoint types the schema forbids. That is joint decoding doing what notebook 03 measured it doing:
typed endpoints are a decoding constraint, so an illegal edge is not filtered out afterwards, it is never
generated. It is the single most reliable property in this repo.

**Recall in the 0.2–0.5 band.** Consistent across domains, and consistent with what notebooks 03 and 07
measured on business news. This pipeline finds roughly a third of a hand-written gold set, and where it is
wrong it is wrong in legible ways.

**The extraction unit matters more than the ontology.** Travel and shopping are documents and extract
directly. Customer service is conversation and needed rewriting plus windowing to reach the same band at all.
Nothing about the ontology predicted that; the shape of the text did.

And one thing that does not transfer:

**Resolution behaviour is corpus-specific and the earlier notebooks over-claimed it.** Travel needed a
dictionary because its identifiers share no characters with their names. Shopping needed something the
similarity resolver does not have, and merged three pairs of distinct products at the same threshold that
scored precision 1.000 in notebooks 01 and 05. "Our resolver has perfect precision" was a statement about
those corpora, not about the resolver.

In [29]:
# Where each domain's edges concentrate -- the graph's shape differs even though
# the machinery does not.
shape = pd.DataFrame([
    {"domain": name,
     **{f"top {i+1}": f"{rel} ({n})"
        for i, (rel, n) in enumerate(
            __import__("collections").Counter(e.type for e in kg.edges).most_common(4))}}
    for name, (_, kg, _, _) in GRAPHS.items()
]).set_index("domain")
display(shape)

,top 1,top 2,top 3,top 4
domain,,,,
travel,located_in (17),operated_by (8),stays_at (8),required_for (5)
customer service,owned_by (17),component_of (12),concerns_product (12),contacted_via (11)
shopping,has_aspect (30),has_specification (19),made_by (12),sold_by (12)


---

## 6. Into Neo4j

All three graphs go into one database. Neo4j Community has exactly one, and
[notebook 02](02_neo4j_graphrag_retrieval.ipynb)'s business-news graph is already living in it, so everything
here is namespaced by domain and removed again at the end.

Namespacing is on the `canon_id`, not just a property: `travel:gaz:LHR` and `shopping:product:aurora-14`
cannot collide even if two ontologies share a label. Each node also carries a `domain` property and a
`__Domain__` label so the whole partition can be selected or dropped in one clause.

In [30]:
import neo4j
from kgx.neo4j_io import (Neo4jConfig, connect, counts, schema_summary,
                          load_graph, ENTITY_LABEL)

config = Neo4jConfig.from_env(uri=os.environ.get("NEO4J_URI", "bolt://localhost:7690"),
                              user=os.environ.get("NEO4J_USER", "neo4j"),
                              password=os.environ.get("NEO4J_PASSWORD", "sandbox-kg"))
if "driver" in globals():
    driver.close()
driver = connect(config, notifications="OFF")

# Clear anything a previous run of THIS notebook left behind before taking the
# baseline, so re-running is idempotent and the teardown check at the end is
# comparing like with like.
DOMAIN_LABEL = "__Domain__"
leftover, _, _ = driver.execute_query(
    f"MATCH (n:{DOMAIN_LABEL}) WITH n, count(*) AS _ DETACH DELETE n RETURN count(*) AS removed")
if leftover and leftover[0]["removed"]:
    print(f"cleared {leftover[0]['removed']} nodes left by a previous run")

baseline = counts(driver)
print(f"connected to {config.uri}")
print(f"already present (notebook 02): {baseline['nodes']} nodes, {baseline['relationships']} relationships")

connected to bolt://localhost:7690
already present (notebook 02): 119 nodes, 241 relationships


In [31]:
from dataclasses import replace as _replace
from kgx.graph import KnowledgeGraph, GraphEdge

def namespaced(kg, domain):
    """Prefix every canonical id so three domains can share one database."""
    remap = {cid: f"{domain}:{cid}" for cid in kg.entities}
    entities = {}
    for cid, ent in kg.entities.items():
        clone = _replace(ent, canon_id=remap[cid])
        clone.attrs = {**ent.attrs, "domain": domain}
        entities[remap[cid]] = clone
    edges = [GraphEdge(remap[e.head], e.type, remap[e.tail], list(e.evidence), e.derived)
             for e in kg.edges if e.head in remap and e.tail in remap]
    return KnowledgeGraph(entities, edges, kg.ontology, dict(kg.stats))

loaded = {}
for name, (onto, kg, _, _) in GRAPHS.items():
    domain = name.replace(" ", "_")
    ns = namespaced(kg, domain)
    stats = load_graph(driver, ns)
    driver.execute_query(
        f"MATCH (n:{ENTITY_LABEL}) WHERE n.canon_id STARTS WITH $p SET n:{DOMAIN_LABEL}, n.domain = $d",
        p=f"{domain}:", d=domain)
    loaded[domain] = ns
    print(f"  {name:18} {stats}")

after = counts(driver)
print(f"\ntotal now: {after['nodes']} nodes, {after['relationships']} relationships")
print(f"  of which this notebook added: {after['nodes'] - baseline['nodes']} nodes, "
      f"{after['relationships'] - baseline['relationships']} relationships")

  travel             {'nodes': 110, 'relationships': 73, 'method': 'dynamic'}


  customer service   {'nodes': 151, 'relationships': 84, 'method': 'dynamic'}


  shopping           {'nodes': 135, 'relationships': 133, 'method': 'dynamic'}

total now: 515 nodes, 531 relationships
  of which this notebook added: 396 nodes, 290 relationships


In [32]:
def q(cypher, **params):
    records, _, _ = driver.execute_query(cypher, routing_=neo4j.RoutingControl.READ, **params)
    return pd.DataFrame([dict(r) for r in records])

display(q("""
MATCH (n:__Domain__)
RETURN n.domain AS domain, count(*) AS nodes,
       count(DISTINCT [l IN labels(n) WHERE NOT l STARTS WITH '__'][0]) AS labels
ORDER BY domain
"""))

display(q("""
MATCH (a:__Domain__)-[r]->(b:__Domain__)
RETURN a.domain AS domain, type(r) AS relationship, count(*) AS n
ORDER BY domain, n DESC
""").groupby("domain").head(3).reset_index(drop=True))

,domain,nodes,labels
0,customer_service,151,13
1,shopping,135,13
2,travel,110,13


,domain,relationship,n
0,customer_service,OWNED_BY,17
1,customer_service,COMPONENT_OF,12
2,customer_service,CONCERNS_PRODUCT,12
3,shopping,HAS_ASPECT,30
4,shopping,HAS_SPECIFICATION,19
5,shopping,MADE_BY,12
6,travel,LOCATED_IN,17
7,travel,STAYS_AT,8
8,travel,OPERATED_BY,8


Queries the graph answers that the source documents do not.

In [33]:
# Travel: every flight a traveller took, with the airline resolved through the gazetteer.
display(q("""
MATCH (t:Traveller)-[:FLIES_ON|BOOKED]->(f)
OPTIONAL MATCH (f)-[:OPERATED_BY]->(al:Airline)
WHERE t.domain = 'travel'
RETURN t.name AS traveller, f.name AS flight_or_booking,
       collect(DISTINCT al.name) AS airline
ORDER BY traveller LIMIT 8
"""))

# Shopping: aspects carrying a polarity, grouped by product.
display(q("""
MATCH (p:Product)-[:HAS_ASPECT]->(a:ProductAspect)
WHERE p.domain = 'shopping'
RETURN p.name AS product, collect(a.name + coalesce(' [' + a.sentiment + ']', ''))[..5] AS aspects
ORDER BY size(aspects) DESC LIMIT 6
"""))

# Customer service: what each issue concerns and who reported it.
display(q("""
MATCH (i:Issue)
WHERE i.domain = 'customer_service'
OPTIONAL MATCH (i)-[:CONCERNS_PRODUCT]->(p:Product)
OPTIONAL MATCH (i)-[:REPORTED_BY]->(c:Customer)
WITH i, collect(DISTINCT p.name) AS products, collect(DISTINCT c.name) AS reporters
WHERE size(products) > 0 OR size(reporters) > 0
RETURN i.name AS issue, products, reporters LIMIT 8
"""))

,traveller,flight_or_booking,airline
0,Elena Vasquez,award tickets,[]
1,John F. Kennedy,+1,[Meridian Air]
2,P. Raman,MR117,[]
3,Priya Raman,MR7K2QX9,[]
4,Tomas Ferreira,Sao Paulo,[]


,product,aspects
0,Aurora-14,"[Shipping, screen, Build quality, Battery ..."
1,Aurora,"[Shipping, Battery, Weight, Storage, returns]"
2,Aurora 14 Pro,"[Shipping, Build quality, Battery life, Cu..."
3,Pro,"[Build quality, Battery life, keyboard]"
4,Halcyon Buds Pro,"[Shipping, Noise cancellation, returns]"
5,N600X,"[Shipping, Build quality]"


,issue,products,reporters
0,shutdown fault,[],[Priya Raman]
1,dropout fault,"[Halcyon Buds, New set]",[Marcus Webb]
2,late dispatch complaint,[Northwind Router N600],[Ana Duarte]
3,stuck pick,[],[Ana Duarte]
4,missed dispatch,[],"[Marcus Webb, Ana Duarte]"
5,tracking,[router],[]
6,dead brick,[],[Marcus Webb]
7,warranty claim,[router],[]


Those three tables are also the clearest view of what is still wrong, which is the argument for putting the
graph in a database rather than reading extraction output.

The travel query shows `P. Raman` and `Priya Raman` as two travellers, `John F. Kennedy` typed as a traveller
rather than an airport, and `Sao Paulo` sitting where a flight should be. The shopping query shows `Aurora`,
`Aurora-14`, `Aurora 14 Pro` and a bare `Pro` as four products — §4.1's resolution failure, now with reviews
attached to the wrong nodes.

None of this is visible in an F1 score. A number tells you how much is wrong; a query tells you *what*, and
these took one clause each.

---

## 7. Drawing them with NVL

[`neo4j-viz`](https://neo4j.com/docs/neo4j-viz/current/) wraps the Neo4j Visualization Library. Two mechanics
carry over from notebook 02 and both matter here.

`from_neo4j` copies **every** property into the render, so anything bulky has to be stripped in Python — a map
projection does not help, because it reads the hydrated graph rather than the returned columns. And each
`render()` call inlines the whole NVL bundle, roughly 8.5 MB, so the count is kept low and every drawing is
also written to `output/` as a standalone page.

In [34]:
from neo4j_viz import VisualizationGraph
from neo4j_viz.neo4j import from_neo4j
from kgx.graph import PALETTE

OUT = ROOT / "output"
OUT.mkdir(exist_ok=True)

def domain_view(domain, *, limit=220):
    """Entity-to-entity subgraph for one domain, styled and stripped."""
    result = driver.execute_query(
        """
        MATCH p = (a:__Domain__)-[r]->(b:__Domain__)
        WHERE a.domain = $d AND b.domain = $d
        RETURN p LIMIT $limit
        """,
        d=domain, limit=limit,
        routing_=neo4j.RoutingControl.READ,
        result_transformer_=neo4j.Result.graph,
    )
    vg = from_neo4j(result)
    for node in vg.nodes:
        for bulky in ("embedding", "docs", "aliases"):
            node.properties.pop(bulky, None)
    vg.set_node_captions(property="name")
    vg.color_nodes(property="type", colors=PALETTE)
    vg.resize_nodes(property="n_mentions", node_radius_min_max=(8, 40))
    return vg

def show(vg, name, **kwargs):
    html = vg.render(**kwargs)
    (OUT / f"{name}.html").write_text(html.data)
    print(f"saved {OUT.name}/{name}.html  ({len(html.data)/1e6:.1f} MB)")
    return html

views = {d: domain_view(d) for d in ("travel", "customer_service", "shopping")}
for d, vg in views.items():
    print(f"  {d:18} {len(vg.nodes):3} nodes  {len(vg.relationships):3} relationships")

  travel              63 nodes   73 relationships
  customer_service    82 nodes   84 relationships
  shopping            85 nodes  133 relationships


In [35]:
show(views["travel"], "nvl_travel", layout="forcedirected", height="620px")

saved output/nvl_travel.html  (8.5 MB)


[NVL interactive render — 8.5 MB of inlined bundle, stripped from the committed file]
Re-run this cell for the live version, or open the standalone copy
it wrote to output/ in a browser.


Node colour is the ontology type, node size is how many times the entity was mentioned. The gazetteer-linked
airports and airlines are the ones with stable ids — hover any of them and the `canon_id` is
`travel:gaz:LHR` rather than a cluster number.

In [36]:
show(views["shopping"], "nvl_shopping", layout="forcedirected", height="620px")

saved output/nvl_shopping.html  (8.6 MB)


[NVL interactive render — 8.6 MB of inlined bundle, stripped from the committed file]
Re-run this cell for the live version, or open the standalone copy
it wrote to output/ in a browser.


The shopping graph draws its own defect. `Aurora`, `Aurora-14` and `Aurora 14 Pro` appear as separate hubs,
each with its own aspects and specifications hanging off it, when three of those should be one node and one
should be a distinct product. A resolution failure is much easier to see as a picture than as a B-cubed
score, which is the practical argument for drawing a graph before trusting it.

In [37]:
# One picture of all three, to make the point that they are three disconnected
# components in a single database rather than one graph.
combined = driver.execute_query(
    """
    MATCH p = (a:__Domain__)-[r]->(b:__Domain__)
    WHERE a.domain = b.domain
    RETURN p LIMIT 400
    """,
    routing_=neo4j.RoutingControl.READ,
    result_transformer_=neo4j.Result.graph,
)
vg = from_neo4j(combined)
for node in vg.nodes:
    for bulky in ("embedding", "docs", "aliases"):
        node.properties.pop(bulky, None)
vg.set_node_captions(property="name")
vg.color_nodes(property="domain", colors=PALETTE)      # colour by DOMAIN this time
vg.resize_nodes(property="n_mentions", node_radius_min_max=(6, 30))
print(f"{len(vg.nodes)} nodes across three domains")
show(vg, "nvl_all_domains", layout="forcedirected", height="680px")

230 nodes across three domains


saved output/nvl_all_domains.html  (8.7 MB)


[NVL interactive render — 8.7 MB of inlined bundle, stripped from the committed file]
Re-run this cell for the live version, or open the standalone copy
it wrote to output/ in a browser.


Three components, no edges between them. That is correct and it is also the limitation: the corpora share
people by name — Priya Raman travels, files support tickets and reviews products — and nothing in this
pipeline connects those, because each domain was resolved in isolation against its own ontology.

Joining them is a cross-domain entity-resolution problem, and it is harder than it looks. `Priya Raman` the
`traveller`, the `customer` and the `review_author` are three types in three ontologies; the type-scoped
blocking that gives this repo's resolver its precision is exactly what prevents the merge. It would need a
type-mapping across ontologies, or a person gazetteer, which is the §2 answer again.

---

## 8. Giving the database back

Everything this notebook wrote is namespaced, so removing it is one clause. Notebook 02's graph and its four
search indexes are left exactly as they were found.

In [38]:
before_teardown = counts(driver)
driver.execute_query(f"MATCH (n:{DOMAIN_LABEL}) DETACH DELETE n")
restored = counts(driver)

print(f"removed {before_teardown['nodes'] - restored['nodes']} nodes, "
      f"{before_teardown['relationships'] - restored['relationships']} relationships")
print(f"\nbaseline was : {baseline['nodes']} nodes, {baseline['relationships']} relationships")
print(f"now          : {restored['nodes']} nodes, {restored['relationships']} relationships")
assert (restored["nodes"], restored["relationships"]) == (baseline["nodes"], baseline["relationships"])

records, _, _ = driver.execute_query(
    "SHOW INDEXES YIELD name, type WHERE type IN ['VECTOR','FULLTEXT'] RETURN collect(name) AS n")
print(f"notebook 02's search indexes : {records[0]['n']}")
driver.close()
print("\ndriver closed")

removed 396 nodes, 290 relationships

baseline was : 119 nodes, 241 relationships
now          : 119 nodes, 241 relationships


notebook 02's search indexes : ['document_ft', 'document_vec', 'entity_ft', 'entity_vec']

driver closed


## What this bought

The method transferred. Three ontologies written for three unrelated domains, three corpora the pipeline had
never seen, and the same 194M encoder produced usable graphs in all three with **zero ontology violations**
and no LLM anywhere.

What did *not* transfer is more useful:

**Resolution is corpus-specific, and the earlier notebooks over-claimed.** B-cubed precision 1.000 in
notebooks 01 and 05 was a property of a corpus with no near-identical entity names, not of the resolver. On
ecommerce the same code at the same threshold merged `Aurora 14` with `Aurora 14 Pro`.

**Where a controlled vocabulary exists, stop computing similarity.** A gazetteer resolved `LHR` to
`London Heathrow` exactly, where Jaro-Winkler scores 0.45 and no threshold reaches it. It also produces stable
ids that survive a rerun and a change of corpus, which clustering cannot.

**A kind constraint has to be validated, not adopted.** The type check between extractor and vocabulary cost
20 points of gazetteer coverage and prevented zero errors, because both sides were reasonable and disagreed
about whether an airport is a place.

**Document-level judgement is the boundary of the no-LLM position.** Intent classification landed near
chance, and priority *was* chance — a near-constant predictor with high confidence. Severity is not written
down, so a span-based model has nothing to key on. Of everything in this notebook, that is the one task worth
escalating.

**The extraction unit is decided by the text, not the ontology.** Documents extract directly; conversations
need first-person rewriting and episode windowing to reach the same recall band. Notebook 01 established the
ordering on agent memory and it reproduced here on support threads written for a different schema — which is
a replication, not a restatement.

## Where to take it

**Cross-domain resolution.** Priya Raman travels, files tickets and writes reviews, and the graph has her as
three unconnected nodes. Type-scoped blocking — the thing that gives this resolver its precision — is exactly
what blocks the merge. A person gazetteer or an explicit type mapping across ontologies would fix it.

**A model-number gazetteer for §4.** Product SKUs are a controlled vocabulary and a catalogue is a published
list. The same technique that fixed travel would fix the `Aurora 14 Pro` merge, and this notebook deliberately
left it out so the failure is on the record.

**A fixed aspect taxonomy.** The aspect-sentiment measurement in §4.2 compares gold against model on whatever
names happen to align, because `screen` and `OLED display` are the same aspect and an exact join sees neither.
Constraining the extractor to a taxonomy, or matching through one, is the right next iteration.

**Escalation, measured.** Notebook 03 sweeps the encoder/LLM escalation rate on business news. The interesting
version here is per-*task* rather than per-document: keep the encoder for extraction, escalate only the
document-level classification, and measure what that costs.